# Experiment 8: error taxonomy

Goal: discover which financial-language phenomena our models get wrong, organise them into an error taxonomy, and use that taxonomy for targeted training (compared against random training and the LLM).

Pipeline:
1. **Error pool**: training sentences that *both* the TF-IDF + LR baseline and the frozen-FinBERT CNN misclassify (out-of-fold, so every prediction comes from a model that never saw the sentence). Errors shared by two different models point at the language, not at one model's quirks.
2. **Manual coding**: each teammate codes a sample of the pool with the categories in the codebook below; a shared subset measures agreement.
3. **Tagging at scale**: rules per category, checked against the hand-coded sentences, applied to all of train and val.
4. **Targeted vs random training per category**, and per-category evaluation (incl. the LLM).

Only the training set is used to build the taxonomy; val and test stay untouched for evaluation.

In [2]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline

sys.path.append("..")
from utils.preprocessing import LABELS, clean_text, encode_labels

pd.set_option("display.max_colwidth", None)
os.makedirs("../results/taxonomy", exist_ok=True)

## 1. Error pool
LR out-of-fold probabilities are recomputed here (same settings as the main baseline). CNN out-of-fold predictions come from experiment 7 (`results/hard_samples/exp7_train_hard.csv`).

In [3]:
train = pd.read_csv("../data/train.csv")
y_train = encode_labels(train.label)

lr = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=clean_text, tokenizer=str.split, token_pattern=None,
                              ngram_range=(1, 2), sublinear_tf=True)),
    ("lr", LogisticRegression(C=10, class_weight="balanced", max_iter=2000)),
])
lr_probs = cross_val_predict(lr, train.sentence, y_train, method="predict_proba",
                             cv=StratifiedKFold(5, shuffle=True, random_state=42))

cnn = pd.read_csv("../results/hard_samples/exp7_train_hard.csv")
assert (cnn.sentence.values == train.sentence.values).all()

errors = train[["sentence", "label"]].copy()
errors.insert(0, "train_idx", np.arange(len(train)))
errors["lr_pred"] = np.array(LABELS)[lr_probs.argmax(axis=1)]
errors["lr_p_true"] = lr_probs[np.arange(len(train)), y_train].round(3)
errors["cnn_pred"] = cnn.pred
errors["cnn_margin"] = cnn.margin.round(3)
errors["n_words"] = train.sentence.str.split().str.len()

both_wrong = (errors.lr_pred != errors.label) & (errors.cnn_pred != errors.label)
pool = errors[both_wrong].copy()
pool["error_type"] = pool.label + " -> " + pool.cnn_pred
pool.to_csv("../results/taxonomy/error_pool.csv", index=False)

print(f"LR wrong: {(errors.lr_pred != errors.label).sum()}, CNN wrong: {(errors.cnn_pred != errors.label).sum()}, "
      f"both wrong (pool): {len(pool)} / {len(train)}")
pool.error_type.value_counts()

LR wrong: 664, CNN wrong: 608, both wrong (pool): 432 / 2901


error_type
positive -> neutral     198
neutral -> positive      81
negative -> neutral      75
negative -> positive     38
neutral -> negative      25
positive -> negative     15
Name: count, dtype: int64

## 2. Codebook (starting version: add categories as you find them)

A sentence can get **several** categories. Mark a category with `1`.

| Column | Phenomenon | Example from the pool / hard set |
|---|---|---|
| `numeric_comparison` | sentiment depends on comparing figures (versus, from X to Y, compared to) | "EPS stood at EUR0.07 versus EUR0.11" |
| `implicit_sentiment` | good/bad news without sentiment words; needs domain knowledge | "Technopolis is Europe's leading operator of technoparks"; "We were the lowest bidders" |
| `object_polarity` | the direction word's polarity depends on its object (costs down = good, sales down = bad) | "restructuring costs of EUR1.6m, while a year earlier they were EUR13.1m" |
| `contrast_negation` | however / but / although / not / still flips or limits the sentiment | "However, the total orders received will still be above last year's levels" |
| `forward_looking` | expectations, plans, hedges (expects, may, will, plans to) | "The effect may remain short-lived" |
| `mixed_sentiment` | positive and negative information in one sentence | profit up but sales down |
| `business_event` | contract, deal, cooperation, expansion, appointment (often unclear whether neutral or positive) | "Ponsse has agreed with Volvo on the start of cooperation in Latin America" |
| `ambiguous_label` | you honestly could argue for another label; no clear phenomenon | "Sanoma will continue to focus on investing in digital media" |
| `other` | none of the above: describe it in `notes` (candidates for new categories) | |

Rules:
- Code what makes the sentence **hard**, not every phenomenon it contains.
- `ambiguous_label` = label noise / disagreement. These are excluded from targeted training later, so use it honestly.
- Do not change the gold label; use `notes` if you think it is wrong.

In [ ]:
CATEGORIES = ["numeric_comparison", "implicit_sentiment", "object_polarity", "contrast_negation",
              "forward_looking", "mixed_sentiment", "business_event", "ambiguous_label", "other"]
CODERS = ["A", "B", "C", "D"]
N_UNIQUE, N_SHARED = 50, 20   # per coder: 50 own sentences + 20 shared by everyone (for agreement)

# Stratified sample over error types, so the coded sentences have the same mix of mistakes as the pool.
sample, _ = train_test_split(pool, train_size=N_SHARED + N_UNIQUE * len(CODERS),
                             stratify=pool.error_type, random_state=42)
sample = sample.sample(frac=1, random_state=42)
shared = sample.iloc[:N_SHARED]
unique = sample.iloc[N_SHARED:]

columns = ["train_idx", "sentence", "label", "cnn_pred", "lr_pred"]
for i, coder in enumerate(CODERS):
    path = f"../results/taxonomy/coding_sheet_{coder}.csv"
    if os.path.exists(path):   # never overwrite a sheet someone may already be coding
        print(f"{path} exists, not overwritten")
        continue
    sheet = pd.concat([shared, unique.iloc[i * N_UNIQUE:(i + 1) * N_UNIQUE]])[columns]
    sheet = sheet.sample(frac=1, random_state=i)          # shared rows mixed in, not all at the top
    sheet["shared"] = sheet.train_idx.isin(shared.train_idx)
    for category in CATEGORIES:
        sheet[category] = ""
    sheet["notes"] = ""
    sheet.to_csv(path, index=False)

print(f"{len(CODERS)} sheets x {N_UNIQUE + N_SHARED} sentences ({N_SHARED} shared), "
      f"{len(sample)} distinct sentences out of {len(pool)} in the pool")
sample.error_type.value_counts()

## 3. Agreement (run after everyone has filled in their sheet)
Pairwise Cohen's kappa per category on the shared sentences, averaged over all coder pairs. Rough guide: > 0.6 good, 0.4-0.6 moderate, < 0.4 the category definition needs work.

In [ ]:
sheets = {c: pd.read_csv(f"../results/taxonomy/coding_sheet_{c}.csv") for c in CODERS}
shared_coded = {c: s[s.shared].set_index("train_idx").sort_index()[CATEGORIES].fillna(0).astype(int)
                for c, s in sheets.items()}

if all(s.to_numpy().sum() > 0 for s in shared_coded.values()):
    pairs = [(a, b) for i, a in enumerate(CODERS) for b in CODERS[i + 1:]]
    kappa = pd.Series({cat: np.mean([cohen_kappa_score(shared_coded[a][cat], shared_coded[b][cat]) for a, b in pairs])
                       for cat in CATEGORIES})
    display(kappa.round(2).sort_values())
else:
    print("Not all sheets are coded yet.")